# Chi² analysis
Le Chi² est pertinent pour tester les associations entre la sévérité, le sexe, la présence d’anomalies cliniques et des seuils lipidiques. Il permet d’évaluer si ces facteurs influencent la distribution des patients dans les groupes étudiés.  
  
Les variables catégorielles que l'on peut analyser avec un test du Chi² sont celles qui classent les patients en groupes distincts.  
  
On considère : 
- ∗ p <0.05. 
- ∗∗ p <0.01 et 
- ∗∗∗ p <0.001

In [203]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import scipy.stats as stats

# Tableaux de contingence des 43 patients

In [306]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/raw_data/Patient_data.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données de l'onglet
patient_data = pd.read_excel(xls, sheet_name="patient_data")
patient_data

,Patient code,Age,Sex,severity,PBMC,Date of fever,Day of Fever
0,BS-064,14,M,severe,1.76,2022-06-18,0
1,JV-035,13,M,severe,11.43,2022-07-05,7
2,KT-515,29,M,severe,1.80,2023-08-24,6
3,KT-926,8,M,severe,6.30,2023-11-09,4
4,JV-148,14,M,severe,5.65,2023-10-23,0
5,JV-157,13,M,severe,7.00,2023-11-08,5
6,KT-193,10,M,severe,1.90,2023-05-13,3
7,KT-247,10,F,severe,2.00,2023-07-02,2
8,KT-312,14,M,severe,1.10,2023-07-18,2
9,KT-347,9,M,severe,6.05,2023-07-28,3


## Statistiques de base

In [307]:
# Age moyen des patients
age_mean = patient_data.Age.mean()
age_std = patient_data.Age.std()
print(f'Age moyen des patients : {age_mean:.2f} ± {age_std:.2f}')
print('Age médian des patients : ', patient_data.Age.median())

Age moyen des patients : 11.22 ± 4.85
Age médian des patients :  10.5


In [240]:
# Calculer le nb et pourcentage de valeurs "M" dans la colonne "Sex"
nb_m = (patient_data.Sex.value_counts()['M'])
percentage_m = (patient_data['Sex'].value_counts(
    normalize=True)['M'] * 100).round(2)

print(
    f"Sex (male) (Total): {nb_m} ({percentage_m}%)")

Sex (male) (Total): 30 (69.77%)


In [308]:
# Filtrer les données par sexe
mild_sex = patient_data[patient_data["severity"] == "mild"]["Sex"]
severe_sex = patient_data[patient_data["severity"] == "severe"]["Sex"]

nb_mild_sex = mild_sex.value_counts()['M']
percentage_mild_sex= (mild_sex.value_counts(normalize=True)['M'] * 100).round(2)

nb_severe_sex = severe_sex.value_counts()['M']
percentage_severe_sex = (severe_sex.value_counts(normalize=True)['M'] * 100).round(2)

print(f"Sex (male) (Mild): {nb_mild_sex} ({percentage_mild_sex:.2f}%)")
print(f"Sex (male) (Severe): {nb_severe_sex} ({percentage_severe_sex:.2f}%)")

Sex (male) (Mild): 12 (54.55%)
Sex (male) (Severe): 15 (83.33%)


## Tableau de contingence Sex / Severity

In [309]:
print(pd.crosstab(patient_data["Sex"], patient_data["severity"]))

severity  mild  severe
Sex                   
F           10       3
M           12      15


In [310]:
# Créer un tableau de contingence
contingency_table = pd.crosstab(patient_data["Sex"], patient_data["severity"])

# Effectuer le test du chi-carré
# Effectuer le test du chi-carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats
print(f"Chi-square value: {chi2:.2f}")
print(f"P-value: {p:.4f}")

# Interprétation des résultats
alpha = 0.05
if p < alpha:
    print("Il existe une association significative entre le sexe et la sévérité.")
else:
    print("Il n'existe pas d'association significative entre le sexe et la sévérité.")

Chi-square value: 2.54
P-value: 0.1108
Il n'existe pas d'association significative entre le sexe et la sévérité.


## Tableau de contingence Age / Severity

In [311]:
# Création de deux groupes : Child et Teen/Adult, en tenant compte de l'âge moyen
patient_data["Age_Group"] = np.where(
    patient_data["Age"] < 11, "Child", "Teen/Adult")

In [312]:
# Tableau de contingence Age_Group et severity
print(pd.crosstab(patient_data["Age_Group"], patient_data["severity"]))

severity    mild  severe
Age_Group               
Child         13       7
Teen/Adult     9      11


In [313]:
# Filtrer les données par sévérité
mild_ages = patient_data[patient_data["severity"] == "mild"]["Age"]
severe_ages = patient_data[patient_data["severity"] == "severe"]["Age"]

mild_ages_mean = mild_ages.mean()
mild_ages_std = mild_ages.std()
severe_ages_mean = severe_ages.mean()
severe_ages_std = severe_ages.std()

# Affichage des résultats
print(
    f"Âge moyen (Mild): {mild_ages_mean:.2f} ± {mild_ages_std:.2f}")
print(
    f"Âge moyen (Severe): {severe_ages_mean:.2f} ± {severe_ages_std:.2f}")

Âge moyen (Mild): 9.86 ± 4.09
Âge moyen (Severe): 12.89 ± 5.30


In [314]:
# Créer un tableau de contingence
contingency_table = pd.crosstab(patient_data["Age"], patient_data["severity"])

# Effectuer le test du chi-carré
# Effectuer le test du chi-carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats
print(f"Chi-square value: {chi2:.2f}")
print(f"P-value: {p:.4f}")

# Interprétation des résultats
alpha = 0.05
if p < alpha:
    print("Il existe une association significative entre l'âge et la sévérité.")
else:
    print("Il n'existe pas d'association significative entre l'âge et la sévérité.")

Chi-square value: 13.15
P-value: 0.5147
Il n'existe pas d'association significative entre l'âge et la sévérité.


In [315]:
t_stat, p_value = stats.ttest_ind(mild_ages, severe_ages, equal_var=False)
print(f"T-test entre Mild et Severe: t = {t_stat:.3f}, p = {p_value:.5f}")

T-test entre Mild et Severe: t = -1.986, p = 0.05578


p > 0.05, la différence d’âge n'est pas statistiquement significative entre les groupes severe et mild.

## Analyse de la relation Nb PBMC (en millions) et sévérité
Le test du Chi² est utilisé pour tester l’indépendance entre deux variables catégorielles. Or, ici :  
	•	“Sévérité” est une variable catégorielle (“mild” / “severe”). ✅  
	•	“PBMC” est une variable continue (valeur en millions). ❌

👉 Le test du Chi² n’est pas adapté car il compare la fréquence des catégories dans un tableau de contingence, et non la relation entre une variable numérique et une variable catégorielle.   
  
Nous allons plutôt réaliser un test T de student ou un test non-paramétrique de Mann & Whitney, après avoir testé la normalité de la distribution des données avec le test de Shapiro.

### Test de Shapiro : normalité des données

In [316]:
# Séparer les PBMC en fonction de la sévérité
pbmc_severe = patient_data[patient_data["severity"] == "severe"]["PBMC"]
pbmc_mild = patient_data[patient_data["severity"] == "mild"]["PBMC"]

pbmc_severe_mean = pbmc_severe.mean()
pbmc_severe_std = pbmc_severe.std()
pbmc_mild_mean = pbmc_mild.mean()
pbmc_mild_std = pbmc_mild.std()

# Test de normalité (Shapiro-Wilk)
stat, p_value_norm_severe = stats.shapiro(pbmc_severe)
print(f"stat for severe: {stat:.2f}, p: {p_value_norm_severe:.5f}")
stat, p_value_norm_mild = stats.shapiro(pbmc_mild)
print(f"stat for mild: {stat:.2f}, p: {p_value_norm_mild:.5f}")

stat for severe: 0.83, p: 0.00376
stat for mild: 0.87, p: 0.00813


	•	Hypothèse nulle (H₀) : Les données suivent une distribution normale.
	•	Hypothèse alternative (H₁) : Les données ne suivent pas une distribution normale.
	•	p-value < 0.05 : On rejette H₀ → les données ne sont pas normales.
	•	p-value ≥ 0.05 : On ne rejette pas H₀ → les données peuvent être considérées comme normales.

In [317]:
print(f"pbmc severe (millions): {pbmc_severe_mean:.2f} ± {pbmc_severe_std:.2f}")
print(f"pbmc mild (millions): {pbmc_mild_mean:.2f} ± {pbmc_mild_std:.2f}")

pbmc severe (millions): 4.14 ± 3.33
pbmc mild (millions): 3.94 ± 2.40


### Test T de Student / Test de Mann-Whitney

In [318]:
# Si les deux groupes suivent une loi normale, on utilise le test t de Student
if p_value_norm_severe > 0.05 and p_value_norm_mild > 0.05:
    t_stat, p_value = stats.ttest_ind(pbmc_severe, pbmc_mild, equal_var=False)
    test_type = "Test t de Student"
else:
    # Sinon, on utilise un test non paramétrique (Mann-Whitney)
    t_stat, p_value = stats.mannwhitneyu(
        pbmc_severe, pbmc_mild, alternative='two-sided')
    test_type = "Test de Mann-Whitney"

print(f"{test_type} : Statistique = {t_stat:.3f}, p-value = {p_value:.3f}")

Test de Mann-Whitney : Statistique = 178.000, p-value = 0.596


Pas de différence significative entre les groupes mild et severe pour les PBMC.  
➝ La distribution des PBMC semble similaire dans les deux groupes.  
➝ Rien ne permet de dire que la sévérité influence la quantité de PBMC.  

# D0 - données non normalisées avec D60

In [41]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/somme_des_espèces_lipides_D0.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [42]:
sum_species

,Family,BS-064D00,BS-082D00,BS-336D00,BS-346D00,BS-351D00,BS-364D00,BS-377D00,BS-671D00,JV-035D00,...,KT-695D00,KT-705D00,KT-716D00,KT-718D00,KT-723D00,KT-771D00,KT-805D00,KT-880D00,KT-926D00,KT-974D00
0,DG,9.534447,7.62489,4.722339,1.558664,8.467026,3.897469,15.211302,4.680918,60.214102,...,4.419748,17.009163,16.755152,13.173997,2.745942,15.849938,23.269528,10.806676,29.573164,2.65621
1,CAR,0.179159,0.618445,0.883822,0.837799,0.68767,1.147424,0.916518,0.861717,0.329057,...,1.114927,0.344134,1.647444,0.178381,1.085803,0.42719,1.9451,0.0,0.439972,1.027213
2,CE,7.760417,2.880691,2.032174,4.008069,4.651314,5.254078,9.4073,1.54218,0.972785,...,2.998043,2.708503,1.755861,2.922082,3.301828,0.0,8.518607,0.0,0.0,4.970909
3,Cer,5.105788,2.08739,1.360267,3.391432,4.487805,4.652435,2.622631,3.137028,2.996562,...,2.292107,2.778303,2.101445,3.01655,2.139231,4.888509,2.245213,1.73043,1.504139,1.3562
4,Chol,11.286518,6.291542,6.413234,14.619689,6.673736,20.374974,7.680811,5.074571,9.816619,...,3.401695,9.944598,8.74197,8.030799,6.272608,9.023244,5.583694,6.580752,3.479196,2.290283
5,HexCer,4.688535,0.0,0.0,0.0,3.391048,1.800382,0.796528,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.237946,0.0,0.0,0.0,0.0
6,LPC,1595.862814,1178.460673,2484.710063,3193.412612,1393.371398,2864.027463,2496.057411,1920.74489,2825.769206,...,1409.941995,1902.790563,4410.176994,2424.034878,2008.677233,2487.394191,4247.985858,1285.414163,2560.284895,856.494209
7,PC,1230.068372,569.089308,994.601819,912.379173,922.631327,872.315089,1293.16717,784.485467,1578.296132,...,523.747302,1104.640678,1112.956729,982.788823,601.696692,1098.566715,1242.601443,944.759301,953.615945,384.049678
8,PE,7.655925,0.0,0.0,0.0,7.782003,14.467283,5.879141,0.0,7.673788,...,4.212518,8.145487,6.093268,0.0,8.997636,13.814243,11.146481,8.501925,0.0,0.0
9,SM,470.149453,197.339077,262.694366,377.081354,305.153037,267.771798,318.375433,287.17671,262.990952,...,244.200976,278.619518,300.04174,230.258755,239.63352,412.261908,379.062052,257.38216,143.147457,160.416971


In [43]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [44]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:-3] for col in sample_cols],
    "Day": [col[-3:] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [45]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D00,BS-064D00,severe
1,BS-082,D00,BS-082D00,mild
2,BS-336,D00,BS-336D00,mild
3,BS-346,D00,BS-346D00,mild
4,BS-351,D00,BS-351D00,mild
5,BS-364,D00,BS-364D00,mild
6,BS-377,D00,BS-377D00,mild
7,BS-671,D00,BS-671D00,mild
8,JV-035,D00,JV-035D00,severe
9,JV-138,D00,JV-138D00,mild


In [46]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [47]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
BS-064D00,BS-064,D00,severe,14,M,severe
BS-082D00,BS-082,D00,mild,18,M,mild
BS-336D00,BS-336,D00,mild,11,F,mild
BS-346D00,BS-346,D00,mild,8,F,mild
BS-351D00,BS-351,D00,mild,7,M,mild
BS-364D00,BS-364,D00,mild,6,M,mild
BS-377D00,BS-377,D00,mild,13,F,mild
BS-671D00,BS-671,D00,mild,8,F,mild
JV-035D00,JV-035,D00,severe,13,M,severe


In [48]:
# Appliquer la transformation log2(x + 0.01) sur les valeurs de lipides
lipid_data = sum_species.iloc[:-1, 1:-1].map(lambda x: np.log2(x + 0.01))
lipid_data

,BS-064D00,BS-082D00,BS-336D00,BS-346D00,BS-351D00,BS-364D00,BS-377D00,BS-671D00,JV-035D00,JV-138D00,...,KT-663D00,KT-695D00,KT-705D00,KT-716D00,KT-718D00,KT-723D00,KT-771D00,KT-805D00,KT-880D00,KT-926D00
0,3.254662,2.932607,2.242554,0.649537,3.083558,1.966234,3.928020,2.229870,5.912269,3.404451,...,0.508182,2.147225,4.089088,4.067394,3.720716,1.462546,3.987315,4.540990,3.435185,4.886704
1,-2.402330,-0.670142,-0.161941,-0.238207,-0.519384,0.210917,-0.110110,-0.198068,-1.560401,-6.643856,...,-1.045158,0.169832,-1.497631,0.728960,-2.408275,0.131989,-1.193668,0.967242,-6.643856,-1.152094
2,2.957992,1.531414,1.030106,2.006502,2.220737,2.396181,3.235314,0.634296,-0.025053,3.318275,...,2.484074,1.588825,1.442812,0.820371,1.551925,1.727628,-6.643856,3.092310,-6.643856,-6.643856
3,2.354957,1.068595,0.454457,1.766142,2.169221,2.221084,1.396506,1.653990,1.588115,2.231918,...,1.227451,1.202955,1.479387,1.078231,1.597674,1.103820,2.292343,1.173264,0.799444,0.598498
4,3.497806,2.655705,2.683300,3.870827,2.740655,4.349434,2.943136,2.346126,3.296695,4.772171,...,3.791213,1.770489,3.315363,3.129608,3.007339,2.651364,3.175244,2.483801,2.720443,1.802895
5,2.232211,-6.643856,-6.643856,-6.643856,1.765979,0.856294,-0.310204,-6.643856,-6.643856,2.246656,...,-0.259251,-6.643856,-6.643856,-6.643856,-6.643856,-6.643856,1.168607,-6.643856,-6.643856,-6.643856
6,10.640130,10.202700,11.278868,11.640888,10.444374,11.483835,11.285441,10.907458,11.464433,11.604441,...,10.941087,10.461430,10.893909,12.106624,11.243201,10.972037,11.280425,12.052567,10.328029,11.322094
7,10.264535,9.152537,9.957990,9.833506,9.849626,9.768722,10.336704,9.615621,10.624161,10.240643,...,9.274100,9.032755,10.109375,10.120195,9.940752,9.232917,10.101420,10.279160,9.883818,9.897280
8,2.938460,-6.643855,-6.643854,-6.643853,2.961994,3.855719,2.558057,-6.643853,2.941818,-6.643854,...,2.955865,2.078103,3.027771,2.609582,-6.643853,3.171149,3.789129,3.479810,3.089485,-6.643855
9,8.877006,7.624606,8.037296,8.558770,8.253436,8.064914,8.314631,8.165845,8.038924,8.759690,...,8.268532,7.931984,8.122204,8.229067,7.847175,7.904746,8.687452,8.566328,8.007824,7.161459


In [49]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane)
thresholds = lipid_data.median(axis=1)
thresholds

0      3.106749
1     -0.562255
2      1.531414
3      1.227451
4      2.831372
5     -6.643856
6     10.907458
7      9.849626
8      1.835982
9      8.037296
10     5.948523
11     8.626705
12     3.711432
dtype: float64

In [50]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,BS-064D00,BS-082D00,BS-336D00,BS-346D00,BS-351D00,BS-364D00,BS-377D00,BS-671D00,JV-035D00,JV-138D00,...,KT-663D00,KT-695D00,KT-705D00,KT-716D00,KT-718D00,KT-723D00,KT-771D00,KT-805D00,KT-880D00,KT-926D00
0,1,0,0,0,0,0,1,0,1,1,...,0,0,1,1,1,0,1,1,1,1
1,0,0,1,1,1,1,1,1,0,0,...,0,1,0,1,0,1,0,1,0,0
2,1,0,0,1,1,1,1,0,0,1,...,1,1,0,0,1,1,0,1,0,0
3,1,0,0,1,1,1,1,1,1,1,...,0,0,1,0,1,0,1,0,0,0
4,1,0,0,1,0,1,1,0,1,1,...,1,0,1,1,1,0,1,0,0,0
5,1,0,1,1,1,1,1,1,0,1,...,1,0,0,0,0,0,1,0,1,0
6,0,0,1,1,0,1,1,0,1,1,...,1,0,0,1,1,1,1,1,0,1
7,1,0,1,0,0,0,1,0,1,1,...,0,0,1,1,1,0,1,1,1,1
8,1,0,0,0,1,1,1,0,1,0,...,1,1,1,1,0,1,1,1,1,0
9,1,0,0,1,1,1,1,1,1,1,...,1,0,1,1,0,0,1,1,0,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [51]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [52]:
chi2_results

{'DG': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046},
 'CAR': {'Chi2': 4.413768796992484, 'p-value': 0.035649996457949557},
 'CE': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'Cer': {'Chi2': 0.0, 'p-value': 1.0},
 'Chol': {'Chi2': 0.0, 'p-value': 1.0},
 'HexCer': {'Chi2': 4.413768796992484, 'p-value': 0.035649996457949557},
 'LPC': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'PC': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'PE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'SM': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'TG': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'FA': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'PI': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046}}

DG, CAR, HexCer et PI montrent une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides pourrait être liée à la gravité de la maladie.

## Test du Chi² sur le sexe en fonction de la sévérité

In [53]:
# Chi² entre le sexe et la sévérité
contingency_sex = pd.crosstab(patient_info["Sex"], patient_info["Severity"])
chi2_sex, p_sex, _, _ = chi2_contingency(contingency_sex)
chi2_results["Sex vs Severity"] = {"Chi2": chi2_sex, "p-value": p_sex}

In [54]:
print(chi2_sex)
print(p_sex)

2.901785714285715
0.0884814827675487


Sex vs Severity a une p-value > 0.05 (0.088), indiquant une faible relation entre le sexe des patients et la sévérité de la maladie. Cela signifie que la sévérité est distribuée ééquitablement selon le sexe.

### Tableau de contingence Sex / Severity

In [58]:
print(pd.crosstab(patient_info["Sex"], patient_info["Severity"]))

Severity  mild  severe
Sex                   
F           10       3
M           11      15


### Résidus standardisés 
Quelle catégorie contribue le plus au Chi²

In [59]:
table = pd.crosstab(patient_info["Sex"], patient_info["Severity"])
chi2, p, dof, expected = chi2_contingency(table)
residuals = (table - expected) / np.sqrt(expected)
print(residuals)

Severity      mild    severe
Sex                         
F         1.133893 -1.224745
M        -0.801784  0.866025


## Test du Chi² sur l'âge en fonction de la sévérité

In [55]:
# Test du Chi² pour l'âge (regroupé en deux catégories)
patient_info["Age_Group"] = np.where(
    patient_info["Age"] < 14, "Child", "Teen/Adult")

# Table de contingence Age vs Sévérité
contingency_table_age = pd.crosstab(
    patient_info["Age_Group"], patient_info["Severity"])
chi2_age, p_age, _, _ = chi2_contingency(contingency_table_age)
chi2_results["Age vs Severity"] = {"Chi2": chi2_age, "p-value": p_age}

# Convertir les résultats en DataFrame
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")

In [56]:
chi2_df

,Chi2,p-value
DG,5.747979,0.016508
CAR,4.413769,0.035650
CE,0.665273,0.414705
Cer,0.000000,1.000000
Chol,0.000000,1.000000
HexCer,4.413769,0.035650
LPC,2.126551,0.144766
PC,3.079558,0.079282
PE,0.029934,0.862640
SM,2.126551,0.144766


Age vs Severity a une p-value < 0.05 (0.034), indiquant une forte relation entre l'âge des patients et la sévérité de la maladie. Cela signifie que la sévérité est distribuée différement selon l'âge.

### Tableau de contingence Age / Severity

In [61]:
print(pd.crosstab(patient_info["Age_Group"], patient_info["Severity"]))

Severity    mild  severe
Age_Group               
Child         19      10
Teen/Adult     2       8


### Calcul des résidus standardisés

In [62]:
table = pd.crosstab(patient_info["Age_Group"], patient_info["Severity"])
chi2, p, dof, expected = chi2_contingency(table)
residuals = (table - expected) / np.sqrt(expected)
print(residuals)

Severity        mild    severe
Age_Group                     
Child       0.856511 -0.925138
Teen/Adult -1.458586  1.575453


In [ ]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


Analyse Chi² terminée. Résultats enregistrés dans chi2_results.csv


In [ ]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_D0.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results.csv")

## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [63]:
from statsmodels.stats.multitest import multipletests

In [64]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

                     Chi2   p-value  p-value (BH corrected)
DG               5.747979  0.016508                0.106950
CAR              4.413769  0.035650                0.106950
CE               0.665273  0.414705                0.518381
Cer              0.000000  1.000000                1.000000
Chol             0.000000  1.000000                1.000000
HexCer           4.413769  0.035650                0.106950
LPC              2.126551  0.144766                0.217149
PC               3.079558  0.079282                0.165903
PE               0.029934  0.862640                0.995353
SM               2.126551  0.144766                0.217149
TG               3.079558  0.079282                0.165903
FA               1.237077  0.266035                0.362775
PI               5.747979  0.016508                0.106950
Sex vs Severity  2.901786  0.088481                0.165903
Age vs Severity  4.502771  0.033840                0.106950


In [67]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_bh_corrected_D0.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D0.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D0.csv


# D3 - données non normalisées avec D60

In [68]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/somme_des_espèces_lipides_D3.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [69]:
sum_species

,Family,BS-064D03,BS-082D03,BS-336D03,BS-346D03,BS-351D03,BS-364D03,BS-377D03,BS-671D03,JV-035D03,...,KT-695D03,KT-705D03,KT-716D03,KT-718D03,KT-723D03,KT-771D03,KT-805D03,KT-880D03,KT-926D03,KT-974D03
0,DG,19.968157,3.117386,16.2149,27.207331,15.631862,17.809934,25.295867,30.261837,21.21604,...,12.142673,8.044282,19.705065,14.141924,25.24184,23.197631,18.775687,8.482933,12.441529,10.414962
1,CAR,0.0,0.0,0.677716,0.708692,0.140242,0.568899,0.222659,0.0,1.844806,...,0.135259,0.43282,0.464531,0.784272,0.741406,0.477466,0.221529,0.0,0.0,0.817418
2,CE,2.848492,0.903338,5.007647,11.814341,6.24188,0.0,0.0,3.369775,3.61475,...,3.825023,6.26406,0.0,5.023619,0.0,0.552063,0.0,0.493274,4.419864,8.515708
3,Cer,4.020806,1.348513,1.793138,7.285975,3.109885,6.035633,0.734463,4.252049,1.172186,...,4.505105,1.731756,2.391915,3.32658,1.361323,3.679226,3.345797,1.168313,5.639939,2.048494
4,Chol,8.622505,3.158353,7.555499,11.876356,7.460192,10.01639,0.0,11.772541,0.0,...,7.479568,11.692061,10.578816,8.876283,5.607217,7.47221,4.415338,2.761311,10.559197,4.831007
5,HexCer,1.200005,0.0,0.995126,0.0,2.132215,0.0,0.0,3.287508,0.0,...,0.0,0.0,1.209474,0.0,0.0,0.981445,0.0,0.0,2.483256,2.44228
6,LPC,3055.235935,1382.943091,2916.41176,2366.958261,2663.136456,2047.918692,3347.278615,2905.570574,3680.441973,...,1522.272879,2603.648706,3725.147573,3352.472096,2337.586571,5423.957621,3231.382883,326.267818,1951.959343,2498.226063
7,PC,1291.379157,487.301511,963.20134,2022.505415,1142.050398,1674.839766,1181.767303,1877.201588,1311.075399,...,990.865207,987.840208,1176.458917,938.548077,994.993911,1057.885482,963.521558,774.630985,903.68168,668.41843
8,PE,6.649396,0.0,0.0,15.422787,0.0,7.623165,0.0,10.557481,0.0,...,5.838379,0.0,0.0,0.0,8.899761,13.340312,0.0,0.0,0.0,4.917157
9,SM,246.5139,140.63322,232.841977,267.130409,258.659715,245.841702,179.422754,300.452895,216.264006,...,265.940885,260.031784,309.759924,294.971223,316.908308,414.619134,258.094079,127.98667,348.684308,217.820758


In [70]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [71]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:-3] for col in sample_cols],
    "Day": [col[-3:] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [72]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D03,BS-064D03,severe
1,BS-082,D03,BS-082D03,mild
2,BS-336,D03,BS-336D03,mild
3,BS-346,D03,BS-346D03,mild
4,BS-351,D03,BS-351D03,mild
5,BS-364,D03,BS-364D03,mild
6,BS-377,D03,BS-377D03,mild
7,BS-671,D03,BS-671D03,mild
8,JV-035,D03,JV-035D03,severe
9,JV-138,D03,JV-138D03,mild


In [73]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [74]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
BS-064D03,BS-064,D03,severe,14,M,severe
BS-082D03,BS-082,D03,mild,18,M,mild
BS-336D03,BS-336,D03,mild,11,F,mild
BS-346D03,BS-346,D03,mild,8,F,mild
BS-351D03,BS-351,D03,mild,7,M,mild
BS-364D03,BS-364,D03,mild,6,M,mild
BS-377D03,BS-377,D03,mild,13,F,mild
BS-671D03,BS-671,D03,mild,8,F,mild
JV-035D03,JV-035,D03,severe,13,M,severe


In [75]:
# Appliquer la transformation log2(x + 0.01) sur les valeurs de lipides
lipid_data = sum_species.iloc[:-1, 1:-1].map(lambda x: np.log2(x + 0.01))
lipid_data

,BS-064D03,BS-082D03,BS-336D03,BS-346D03,BS-351D03,BS-364D03,BS-377D03,BS-671D03,JV-035D03,JV-138D03,...,KT-663D03,KT-695D03,KT-705D03,KT-716D03,KT-718D03,KT-723D03,KT-771D03,KT-805D03,KT-880D03,KT-926D03
0,4.320352,1.644958,4.020138,4.766454,3.967340,4.155420,4.661400,4.919904,4.407763,3.810856,...,3.537231,3.603202,3.009756,4.301227,3.822926,4.658317,4.536527,4.231562,3.086263,3.638251
1,-6.643856,-6.643856,-0.540114,-0.476554,-2.734639,-0.788617,-2.103709,-6.643856,0.891268,-6.643856,...,-6.643856,-2.783303,-1.175208,-1.075425,-0.332295,-0.412335,-1.036626,-2.110738,-6.643856,-6.643856
2,1.515254,-0.130779,2.327011,3.563688,2.644290,-6.643856,-6.643856,1.756927,1.857881,-6.643856,...,-1.040649,1.939235,2.649399,-6.643856,2.331596,-6.643856,-0.831195,-6.643856,-0.990585,2.147262
3,2.011068,0.442028,0.850510,2.867101,1.641493,2.595893,-0.425727,2.091547,0.241457,1.518062,...,1.056041,2.174760,0.800543,1.264185,1.738370,0.455569,1.883318,1.746655,0.236723,2.498235
4,3.109779,1.663733,2.919435,3.571235,2.901145,3.325730,-6.643856,3.558579,-6.643856,2.989791,...,3.197970,2.904882,3.548691,3.404469,3.151580,2.489856,2.903464,2.145788,1.470569,3.401794
5,0.275013,-6.643856,0.007377,-6.643856,1.099103,-6.643856,-6.643856,1.721376,-6.643856,0.414607,...,-6.643856,-6.643856,-6.643856,0.286259,-6.643856,-6.643856,-0.012395,-6.643856,-6.643856,1.318031
6,11.577073,10.433537,11.509984,11.208825,11.378916,10.999950,11.708777,11.504611,11.845667,10.574487,...,8.569690,10.572021,11.346325,11.863086,11.711014,11.190810,12.405133,11.657940,8.349957,10.930715
7,10.334708,8.928700,9.911709,10.981935,10.157423,10.709816,10.206742,10.874376,10.356546,9.968235,...,10.141553,9.952560,9.948148,10.200247,9.874302,9.958558,10.046981,9.912188,9.597384,9.819687
8,2.735391,-6.643850,-6.643854,3.947927,-6.643853,2.932281,-6.643854,3.401560,-6.643852,2.041658,...,4.302206,2.548037,-6.643853,-6.643854,-6.643854,3.155387,3.738802,-6.643853,-6.643854,-6.643854
9,7.945584,7.135896,7.863269,8.061454,8.014967,7.941645,7.487299,8.231043,7.756716,7.827887,...,8.756047,8.055016,8.022600,8.275053,8.204479,8.307967,8.695678,8.011809,6.999962,8.445819


In [76]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane)
thresholds = lipid_data.median(axis=1)
thresholds

0      4.155420
1     -1.988607
2     -0.130779
3      1.441680
4      2.903464
5     -6.643856
6     11.318400
7      9.968235
8     -6.643852
9      7.941645
10     7.066998
11     8.180973
12     4.766097
dtype: float64

In [77]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,BS-064D03,BS-082D03,BS-336D03,BS-346D03,BS-351D03,BS-364D03,BS-377D03,BS-671D03,JV-035D03,JV-138D03,...,KT-663D03,KT-695D03,KT-705D03,KT-716D03,KT-718D03,KT-723D03,KT-771D03,KT-805D03,KT-880D03,KT-926D03
0,1,0,0,1,0,0,1,1,1,0,...,0,0,0,1,0,1,1,1,0,0
1,0,0,1,1,0,1,0,0,1,0,...,0,0,1,1,1,1,1,0,0,0
2,1,0,1,1,1,0,0,1,1,0,...,0,1,1,0,1,0,0,0,0,1
3,1,0,0,1,1,1,0,1,0,1,...,0,1,0,0,1,0,1,1,0,1
4,1,0,1,1,0,1,0,1,0,1,...,1,1,1,1,1,0,0,0,0,1
5,1,1,1,1,1,1,0,1,1,1,...,0,0,0,1,0,0,1,1,0,1
6,1,0,1,0,1,0,1,1,1,0,...,0,0,1,1,1,0,1,1,0,0
7,1,0,0,1,1,1,1,1,1,0,...,1,0,0,1,0,0,1,0,0,0
8,1,1,0,1,0,1,0,1,0,1,...,1,1,0,0,0,1,1,0,0,0
9,1,0,0,1,1,0,0,1,0,0,...,1,1,1,1,1,1,1,1,0,1


## Test du Chi² sur les lipides en fonction de la sévérité

In [78]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [79]:
chi2_results

{'DG': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'CAR': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'CE': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'Cer': {'Chi2': 0.0, 'p-value': 1.0},
 'Chol': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'HexCer': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'LPC': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'PC': {'Chi2': 0.0, 'p-value': 1.0},
 'PE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'SM': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'TG': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'FA': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PI': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848}}

Aucun lipide ne montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides ne seraient pas liés à la gravité de la maladie.

In [86]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [87]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_D3.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D3.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [88]:
from statsmodels.stats.multitest import multipletests

In [89]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

                     Chi2   p-value  p-value (BH corrected)
DG               0.220536  0.638632                0.993428
CAR              0.220536  0.638632                0.993428
CE               1.237077  0.266035                0.620749
Cer              0.000000  1.000000                1.000000
Chol             0.029934  0.862640                1.000000
HexCer           2.126551  0.144766                0.620749
LPC              1.237077  0.266035                0.620749
PC               0.000000  1.000000                1.000000
PE               0.029934  0.862640                1.000000
SM               0.029934  0.862640                1.000000
TG               3.079558  0.079282                0.619370
FA               0.220536  0.638632                0.993428
PI               1.237077  0.266035                0.620749
Sex vs Severity  2.901786  0.088481                0.619370


In [90]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_bh_corrected_D3.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D3.csv


# D10 - données non normalisées avec D60

In [91]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/somme_des_espèces_lipides_D10.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [92]:
sum_species

,Family,BS-064D10,BS-082D10,BS-336D10,BS-346D10,BS-351D10,BS-364D10,BS-377D10,BS-671D10,JV-035D10,...,KT-695D10,KT-705D10,KT-716D10,KT-718D10,KT-723D10,KT-771D10,KT-805D10,KT-880D10,KT-926D10,KT-974D10
0,DG,17.14387,8.932488,16.818381,43.06756,20.678015,17.238723,22.828113,10.631183,31.610864,...,15.974618,12.078603,4.522523,24.089838,21.538093,29.826527,39.65641,27.917212,12.63445,7.077127
1,CAR,0.0,0.157989,0.13242,0.587202,0.0,0.876243,0.12918,0.139246,0.0,...,0.562671,0.241541,0.09994,0.112721,0.0,0.242009,0.125143,0.333768,0.0,0.0
2,CE,0.0,3.49802,5.680665,0.0,0.818111,3.075995,0.596266,1.50148,0.0,...,0.492665,1.10157,4.74578,3.433631,4.235984,4.140759,2.38146,0.0,0.0,0.0
3,Cer,2.69399,1.487014,2.639436,2.032787,2.761311,1.98128,1.985246,1.4586,1.492553,...,2.293471,2.2072,1.831182,3.825734,4.000152,2.114922,2.276248,1.794931,2.856815,2.011308
4,Chol,7.239538,3.166453,6.545276,8.051377,6.853948,7.724413,5.896456,9.15952,0.0,...,5.212779,8.712769,10.467162,12.170495,9.741615,11.937205,4.308602,8.229098,4.371859,5.124208
5,HexCer,0.0,0.0,1.181572,0.0,0.0,1.351111,0.0,0.0,0.0,...,0.0,0.0,0.0,0.747076,0.936774,0.724351,0.0,0.0,0.0,0.0
6,LPC,5249.176174,2382.796609,4491.96769,5931.872054,3355.32854,5447.62691,3905.996284,4447.933695,3203.237789,...,5987.756741,4549.63197,3914.471619,5134.018605,4351.614278,4693.378181,4365.95225,2537.055866,3808.740103,3253.496498
7,PC,1853.936574,696.448882,1496.670193,2685.324364,1423.513974,1445.205676,1564.542329,1014.618086,1554.976813,...,1354.029404,1764.619306,1345.198729,2243.03347,1047.403808,1673.373085,1183.84809,2104.073344,1042.136267,1115.957729
8,PE,9.021984,0.0,0.0,13.728119,0.0,0.0,9.885526,0.0,0.0,...,5.371785,5.496055,9.002522,13.799677,0.0,29.284946,0.0,14.831093,0.0,0.0
9,SM,242.426089,107.55478,228.352511,240.467079,174.398995,259.471558,182.578614,257.435066,191.389987,...,189.611506,250.551399,279.93824,278.262692,231.335389,296.21825,181.289846,352.295045,192.032251,181.889737


In [93]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [94]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:-3] for col in sample_cols],
    "Day": [col[-3:] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [95]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D10,BS-064D10,severe
1,BS-082,D10,BS-082D10,mild
2,BS-336,D10,BS-336D10,mild
3,BS-346,D10,BS-346D10,mild
4,BS-351,D10,BS-351D10,mild
5,BS-364,D10,BS-364D10,mild
6,BS-377,D10,BS-377D10,mild
7,BS-671,D10,BS-671D10,mild
8,JV-035,D10,JV-035D10,severe
9,JV-138,D10,JV-138D10,mild


In [96]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [97]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
BS-064D10,BS-064,D10,severe,14,M,severe
BS-082D10,BS-082,D10,mild,18,M,mild
BS-336D10,BS-336,D10,mild,11,F,mild
BS-346D10,BS-346,D10,mild,8,F,mild
BS-351D10,BS-351,D10,mild,7,M,mild
BS-364D10,BS-364,D10,mild,6,M,mild
BS-377D10,BS-377,D10,mild,13,F,mild
BS-671D10,BS-671,D10,mild,8,F,mild
JV-035D10,JV-035,D10,severe,13,M,severe


In [98]:
# Appliquer la transformation log2(x + 0.01) sur les valeurs de lipides
lipid_data = sum_species.iloc[:-1, 1:-1].map(lambda x: np.log2(x + 0.01))
lipid_data

,BS-064D10,BS-082D10,BS-336D10,BS-346D10,BS-351D10,BS-364D10,BS-377D10,BS-671D10,JV-035D10,JV-138D10,...,KT-663D10,KT-695D10,KT-705D10,KT-716D10,KT-718D10,KT-723D10,KT-771D10,KT-805D10,KT-880D10,KT-926D10
0,4.100462,3.160676,4.072824,5.428865,4.370723,4.108418,4.513372,3.411587,4.982805,5.349059,...,4.695467,3.998612,3.595576,2.180314,4.590952,4.429488,4.899008,5.309846,4.803600,3.660432
1,-6.643856,-2.573561,-2.811781,-0.743708,-6.643856,-0.174227,-2.844978,-2.744239,-6.643856,-6.643856,...,-6.643856,-0.804222,-1.991133,-3.185213,-3.026550,-6.643856,-1.988454,-2.887445,-1.540491,-6.643856
2,-6.643856,1.810657,2.508597,-6.643856,-0.272105,1.625736,-0.721977,0.595962,-6.643856,-0.211547,...,-6.643856,-0.992330,0.152598,2.249682,1.783931,2.086099,2.053375,1.257892,-6.643856,-6.643856
3,1.435090,0.582087,1.405685,1.030539,1.470569,0.993696,0.996566,0.554441,0.587416,1.815228,...,2.229638,1.203810,1.148739,0.880632,1.939503,2.003657,1.087410,1.192982,0.851944,1.519449
4,2.857889,1.667417,2.712656,3.011026,2.779039,2.951292,2.562293,3.196846,-6.643856,2.604255,...,3.807534,2.384818,3.124786,3.389176,3.606501,3.285641,3.578601,2.110564,3.042486,2.131543
5,-6.643856,-6.643856,0.252866,-6.643856,-6.643856,0.444785,-6.643856,-6.643856,-6.643856,1.060255,...,-6.643856,-6.643856,-6.643856,-6.643856,-0.401490,-0.078908,-0.445458,-6.643856,-6.643856,-6.643856
6,12.357878,11.218446,12.133135,12.534274,11.712243,12.411415,11.931479,12.118923,11.645320,11.992921,...,12.453829,12.547802,12.151537,11.934606,12.325876,12.087338,12.196414,12.092084,11.308945,11.895102
7,10.856384,9.443894,10.547550,11.390886,10.475251,10.497069,10.611534,9.986735,10.602687,11.117754,...,10.632331,10.403054,10.785149,10.393614,11.131242,10.032616,10.708552,10.209280,11.038976,10.025342
8,3.175043,-6.643854,-6.643854,3.780113,-6.643854,-6.643853,3.306776,-6.643853,-6.643852,4.491304,...,-6.643853,2.428085,2.461019,3.171931,3.787608,-6.643854,4.872580,-6.643854,3.891525,-6.643853
9,7.921461,6.749062,7.835182,7.909756,7.446331,8.019488,7.512453,8.008121,7.580447,7.939963,...,8.028507,7.566979,7.969020,8.129016,8.120356,7.853905,8.210565,7.502234,8.460681,7.585280


In [99]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane)
thresholds = lipid_data.median(axis=1)
thresholds

0      4.331233
1     -3.185213
2      0.548317
3      1.127642
4      2.891119
5     -6.643856
6     12.133135
7     10.547550
8     -6.643851
9      7.835182
10     7.022551
11     7.431276
12     5.634490
dtype: float64

In [100]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,BS-064D10,BS-082D10,BS-336D10,BS-346D10,BS-351D10,BS-364D10,BS-377D10,BS-671D10,JV-035D10,JV-138D10,...,KT-663D10,KT-695D10,KT-705D10,KT-716D10,KT-718D10,KT-723D10,KT-771D10,KT-805D10,KT-880D10,KT-926D10
0,0,0,0,1,1,0,1,0,1,1,...,1,0,0,0,1,1,1,1,1,0
1,0,1,1,1,0,1,1,1,0,0,...,0,1,1,0,1,0,1,1,1,0
2,0,1,1,0,0,1,0,1,0,0,...,0,0,0,1,1,1,1,1,0,0
3,1,0,1,0,1,0,0,0,0,1,...,1,1,1,0,1,1,0,1,0,1
4,0,0,0,1,0,1,0,1,0,0,...,1,0,1,1,1,1,1,0,1,0
5,1,0,1,1,0,1,0,1,1,1,...,1,0,0,0,1,1,1,0,1,1
6,1,0,0,1,0,1,0,0,0,0,...,1,1,1,0,1,0,1,0,0,0
7,1,0,0,1,0,0,1,0,1,1,...,1,0,1,0,1,0,1,0,1,0
8,1,0,0,1,0,0,1,0,0,1,...,0,1,1,1,1,0,1,0,1,0
9,1,0,0,1,0,1,0,1,0,1,...,1,0,1,1,1,1,1,0,1,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [101]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [102]:
chi2_results

{'DG': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'CAR': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'CE': {'Chi2': 0.0, 'p-value': 1.0},
 'Cer': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'Chol': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'HexCer': {'Chi2': 0.0, 'p-value': 1.0},
 'LPC': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PC': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'PE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'SM': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'TG': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'FA': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'PI': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531}}

Aucun lipide ne montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides ne seraient pas liés à la gravité de la maladie.

In [103]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [104]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_D10.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D10.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D10.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [105]:
from statsmodels.stats.multitest import multipletests

In [106]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

            Chi2   p-value  p-value (BH corrected)
DG      0.029934  0.862640                     1.0
CAR     2.126551  0.144766                     1.0
CE      0.000000  1.000000                     1.0
Cer     1.237077  0.266035                     1.0
Chol    0.665273  0.414705                     1.0
HexCer  0.000000  1.000000                     1.0
LPC     0.220536  0.638632                     1.0
PC      1.237077  0.266035                     1.0
PE      0.029934  0.862640                     1.0
SM      0.029934  0.862640                     1.0
TG      0.029934  0.862640                     1.0
FA      0.665273  0.414705                     1.0
PI      0.220536  0.638632                     1.0


In [107]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_bh_corrected_D10.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D10.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D10.csv


# D0 - données normalisées (log2FC) avec D60

In [362]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D0.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [363]:
sum_species

,Family,log2FC_D00-BS-064,log2FC_D00-BS-082,log2FC_D00-BS-336,log2FC_D00-BS-346,log2FC_D00-BS-351,log2FC_D00-BS-364,log2FC_D00-BS-377,log2FC_D00-BS-671,log2FC_D00-JV-035,...,log2FC_D00-KT-695,log2FC_D00-KT-705,log2FC_D00-KT-716,log2FC_D00-KT-718,log2FC_D00-KT-723,log2FC_D00-KT-771,log2FC_D00-KT-805,log2FC_D00-KT-880,log2FC_D00-KT-926,log2FC_D00-KT-974
0,DG,-0.4221192235003327,-0.07095162803153392,-1.5319512047999466,-3.493467806616883,0.08495628379781153,-1.0195638425279017,-0.9704362744718289,-1.3042116312498873,2.5426994038362527,...,-0.3070950141149235,-0.1426981273969576,0.22406169446698349,0.6924002827741179,-1.8629529444520128,-1.2567914172116847,0.6364121058181116,-0.8211265936958543,2.0453619395853573,-1.4103604614963596
1,CAR,-0.7378244554648428,-0.5560028568155733,0.4579879730255637,0.3672571978562007,-0.2398698462216052,0.14008625190855203,30.713194422459736,1.1893551904306594,29.2068794156973,...,30.009036542551325,-0.5580472228849015,1.2076867105301932,-1.6124447052226634,0.4145508567004543,-0.748516930757655,2.108703405007343,-27.305830274094863,1.7363871285932746,30.863099388837714
2,CE,2.203709544727422,-0.883650383910036,-0.09141040636605446,0.8256700874471666,2.3712110251304788,1.0458485473140955,3.1047892695802024,0.13893393238864518,-3.025191214835925,...,-0.9202605845289551,-0.15392727867143519,-2.2854192316231483,0.2393410388534318,-1.9530876060958458,-0.5339105107181525,2.2932134163207025,-0.3108794190715072,-32.181675683263435,1.8448860020001157
3,Cer,1.2386046317024206,-0.20634466941725968,0.6586802683672264,1.611260689901833,31.232860278543995,1.4401337972386985,2.2618331410954053,1.08455817007167,1.565922576186252,...,0.03839896420465231,-0.698715658165472,0.6103329130704763,0.09350816802539849,-0.21083534295647902,1.061575475885075,0.05161859975031745,1.179902411928127,-0.34631384288389416,0.3956128696981117
4,Chol,-0.22168270950936883,-0.21178600247656196,-1.1355066929358573,-0.593531857995445,0.52670878325349,0.4938871007335568,0.6978442254771307,-0.5849498504662766,-0.003763868150014045,...,-0.6304180141840532,0.5375673636348789,1.0476007068956072,0.21927737631565986,-0.5037958360539728,0.9410267552598676,0.004759268546097433,-0.183247025161631,-0.5751810914262777,33.18051103749568
5,HexCer,31.141987688278313,-30.79583032540669,0.2599097056925564,-28.552595260211778,30.691043717844178,30.278122518615287,29.07178516891364,-29.21351066343342,0.4699721046457717,...,-1.1319011992162573,-30.877395823651675,-29.116429402749617,-0.2361418314444066,-0.24843455517443944,30.15659918802953,-29.20968840475806,0.19725098678845118,-29.93624449272328,-0.01582496716168297
6,LPC,-1.3814670387958732,-1.9739118416143338,-0.9043681954323097,-1.1608989295482917,-1.6635211621471084,-0.2590012610656178,-0.024718015319778686,-0.600386940595662,-0.6090474931500824,...,-1.4635652969721502,-1.4401078897617237,-0.3455966982722982,-0.8195143022529598,-1.0555949583545947,-1.0160499865211297,0.3902714974822787,-0.9324354427590325,-0.189193345232874,-2.1658138405538603
7,PC,0.29600505551478407,-1.2961513016841515,-0.2719173541840447,-0.6026066126844256,-0.3301519710411789,-0.006344023570147133,0.43146840602597153,-0.1903930051125798,0.49968384005813127,...,-0.9793172970573618,-0.3934951815221625,0.061522904482061526,-0.4503332997834732,-0.58860886820517,-0.32108496620676896,0.10460116826581342,-0.9956521265152739,-0.10160048285492998,-1.9058324302119607
8,PE,28.47910288725836,-30.29523925978197,0.2742773261982051,0.35198976450909136,28.80586861799048,0.605541561868752,29.119618328275237,1.1608134722209489,28.779272171888667,...,26.783949968445732,28.952094385815528,28.421427923322856,-28.75004278636596,28.78922708751448,-0.8122854253885701,-0.04341117579067182,-0.5853322460832162,-31.081515868896577,-1.3184971894870892
9,SM,1.0335694518443708,-0.42137938586927415,-0.007939915093905727,0.2175818791013914,0.7151266896289435,0.38670111366095916,1.2958916435767787,0.411911450907579,0.39817899578620086,...,-0.12005559835470583,-0.11

In [364]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,KT-515,29,M,severe
3,KT-926,8,M,severe
4,JV-148,14,M,severe
5,JV-157,13,M,severe
6,KT-193,10,M,severe
7,KT-247,10,F,severe
8,KT-312,14,M,severe
9,KT-347,9,M,severe


In [370]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[11:] for col in sample_cols],
    "Day": [col[7:-7] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:].values
})
patient_info

,Sample,Day,Column,Severity
0,BS-064,D00,log2FC_D00-BS-064,severe
1,BS-082,D00,log2FC_D00-BS-082,mild
2,BS-336,D00,log2FC_D00-BS-336,mild
3,BS-346,D00,log2FC_D00-BS-346,mild
4,BS-351,D00,log2FC_D00-BS-351,mild
5,BS-364,D00,log2FC_D00-BS-364,mild
6,BS-377,D00,log2FC_D00-BS-377,mild
7,BS-671,D00,log2FC_D00-BS-671,mild
8,JV-035,D00,log2FC_D00-JV-035,severe
9,JV-138,D00,log2FC_D00-JV-138,mild


In [371]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
log2FC_D00-BS-064,BS-064,D00,severe,14,M,severe
log2FC_D00-BS-082,BS-082,D00,mild,18,M,mild
log2FC_D00-BS-336,BS-336,D00,mild,11,F,mild
log2FC_D00-BS-346,BS-346,D00,mild,8,F,mild
log2FC_D00-BS-351,BS-351,D00,mild,7,M,mild
log2FC_D00-BS-364,BS-364,D00,mild,6,M,mild
log2FC_D00-BS-377,BS-377,D00,mild,13,F,mild
log2FC_D00-BS-671,BS-671,D00,mild,8,F,mild
log2FC_D00-JV-035,JV-035,D00,severe,13,M,severe


In [372]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [373]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0    -0.307095
1     1.337141
2     0.239341
3     0.711893
4     0.178193
5    -0.243950
6    -0.819514
7    -0.271917
8     0.351990
9     0.330232
10   -0.360692
11    1.474163
12   -1.162457
dtype: float64

In [374]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,log2FC_D00-BS-064,log2FC_D00-BS-082,log2FC_D00-BS-336,log2FC_D00-BS-346,log2FC_D00-BS-351,log2FC_D00-BS-364,log2FC_D00-BS-377,log2FC_D00-BS-671,log2FC_D00-JV-035,log2FC_D00-JV-138,...,log2FC_D00-KT-663,log2FC_D00-KT-695,log2FC_D00-KT-705,log2FC_D00-KT-716,log2FC_D00-KT-718,log2FC_D00-KT-723,log2FC_D00-KT-771,log2FC_D00-KT-805,log2FC_D00-KT-880,log2FC_D00-KT-926
0,0,1,0,0,1,0,0,0,1,0,...,0,0,1,1,1,0,0,1,0,1
1,0,0,0,0,0,0,1,0,1,0,...,1,1,0,0,0,0,0,1,0,1
2,1,0,0,1,1,1,1,0,0,1,...,1,0,0,0,0,0,0,1,0,0
3,1,0,0,1,1,1,1,1,1,1,...,1,0,0,0,0,0,1,0,1,0
4,0,0,0,0,1,1,1,0,0,1,...,1,0,1,1,1,0,1,0,0,0
5,1,0,1,0,1,1,1,0,1,1,...,1,0,0,0,1,0,1,0,1,0
6,0,0,0,0,0,1,1,1,1,1,...,1,0,0,1,0,0,0,1,0,1
7,1,0,0,0,0,1,1,1,1,1,...,0,0,0,1,0,0,0,1,0,1
8,1,0,0,0,1,1,1,1,1,0,...,0,1,1,1,0,1,0,0,0,0
9,1,0,0,0,1,1,1,1,1,1,...,1,0,0,0,0,0,1,1,0,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [375]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}
chi2_results

{'DG': {'Chi2': 9.242340225563906, 'p-value': 0.002364825341650665},
 'CAR': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'CE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'Cer': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'Chol': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'HexCer': {'Chi2': 0.0, 'p-value': 1.0},
 'LPC': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PC': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'PE': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'SM': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'TG': {'Chi2': 9.242340225563906, 'p-value': 0.002364825341650665},
 'FA': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PI': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046}}

DG, TG et PI montrent une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides seraient liés à la gravité de la maladie. Reste à voir si cette valeur passe la correction de Benjamini-Hochberg.

In [376]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")
chi2_df


,Chi2,p-value
DG,9.242340,0.002365
CAR,0.220536,0.638632
CE,0.029934,0.862640
Cer,0.029934,0.862640
Chol,0.220536,0.638632
HexCer,0.000000,1.000000
LPC,0.220536,0.638632
PC,3.079558,0.079282
PE,0.220536,0.638632
SM,0.029934,0.862640


In [377]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_log2FC_D0.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D0.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D0.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [378]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

            Chi2   p-value  p-value (BH corrected)
DG      9.242340  0.002365                0.015371
CAR     0.220536  0.638632                0.922468
CE      0.029934  0.862640                0.934526
Cer     0.029934  0.862640                0.934526
Chol    0.220536  0.638632                0.922468
HexCer  0.000000  1.000000                1.000000
LPC     0.220536  0.638632                0.922468
PC      3.079558  0.079282                0.257667
PE      0.220536  0.638632                0.922468
SM      0.029934  0.862640                0.934526
TG      9.242340  0.002365                0.015371
FA      0.220536  0.638632                0.922468
PI      5.747979  0.016508                0.071533


DG, TG passent avec succès la correction de Benjamini-Hochberg.

In [342]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_log2FC_D0.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D0.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D0.csv


## Ratio PC/PE vs sévérité

In [379]:
import pandas as pd

# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D0.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données en excluant la dernière ligne de sum_species
sum_species = pd.read_excel(xls, sheet_name="sum_species").iloc[:-1]
sum_species

,Family,log2FC_D00-BS-064,log2FC_D00-BS-082,log2FC_D00-BS-336,log2FC_D00-BS-346,log2FC_D00-BS-351,log2FC_D00-BS-364,log2FC_D00-BS-377,log2FC_D00-BS-671,log2FC_D00-JV-035,...,log2FC_D00-KT-695,log2FC_D00-KT-705,log2FC_D00-KT-716,log2FC_D00-KT-718,log2FC_D00-KT-723,log2FC_D00-KT-771,log2FC_D00-KT-805,log2FC_D00-KT-880,log2FC_D00-KT-926,log2FC_D00-KT-974
0,DG,-0.4221192235003327,-0.07095162803153392,-1.5319512047999466,-3.493467806616883,0.08495628379781153,-1.0195638425279017,-0.9704362744718289,-1.3042116312498873,2.5426994038362527,...,-0.3070950141149235,-0.1426981273969576,0.22406169446698349,0.6924002827741179,-1.8629529444520128,-1.2567914172116847,0.6364121058181116,-0.8211265936958543,2.0453619395853573,-1.4103604614963596
1,CAR,-0.7378244554648428,-0.5560028568155733,0.4579879730255637,0.3672571978562007,-0.2398698462216052,0.14008625190855203,30.713194422459736,1.1893551904306594,29.2068794156973,...,30.009036542551325,-0.5580472228849015,1.2076867105301932,-1.6124447052226634,0.4145508567004543,-0.748516930757655,2.108703405007343,-27.305830274094863,1.7363871285932746,30.863099388837714
2,CE,2.203709544727422,-0.883650383910036,-0.09141040636605446,0.8256700874471666,2.3712110251304788,1.0458485473140955,3.1047892695802024,0.13893393238864518,-3.025191214835925,...,-0.9202605845289551,-0.15392727867143519,-2.2854192316231483,0.2393410388534318,-1.9530876060958458,-0.5339105107181525,2.2932134163207025,-0.3108794190715072,-32.181675683263435,1.8448860020001157
3,Cer,1.2386046317024206,-0.20634466941725968,0.6586802683672264,1.611260689901833,31.232860278543995,1.4401337972386985,2.2618331410954053,1.08455817007167,1.565922576186252,...,0.03839896420465231,-0.698715658165472,0.6103329130704763,0.09350816802539849,-0.21083534295647902,1.061575475885075,0.05161859975031745,1.179902411928127,-0.34631384288389416,0.3956128696981117
4,Chol,-0.22168270950936883,-0.21178600247656196,-1.1355066929358573,-0.593531857995445,0.52670878325349,0.4938871007335568,0.6978442254771307,-0.5849498504662766,-0.003763868150014045,...,-0.6304180141840532,0.5375673636348789,1.0476007068956072,0.21927737631565986,-0.5037958360539728,0.9410267552598676,0.004759268546097433,-0.183247025161631,-0.5751810914262777,33.18051103749568
5,HexCer,31.141987688278313,-30.79583032540669,0.2599097056925564,-28.552595260211778,30.691043717844178,30.278122518615287,29.07178516891364,-29.21351066343342,0.4699721046457717,...,-1.1319011992162573,-30.877395823651675,-29.116429402749617,-0.2361418314444066,-0.24843455517443944,30.15659918802953,-29.20968840475806,0.19725098678845118,-29.93624449272328,-0.01582496716168297
6,LPC,-1.3814670387958732,-1.9739118416143338,-0.9043681954323097,-1.1608989295482917,-1.6635211621471084,-0.2590012610656178,-0.024718015319778686,-0.600386940595662,-0.6090474931500824,...,-1.4635652969721502,-1.4401078897617237,-0.3455966982722982,-0.8195143022529598,-1.0555949583545947,-1.0160499865211297,0.3902714974822787,-0.9324354427590325,-0.189193345232874,-2.1658138405538603
7,PC,0.29600505551478407,-1.2961513016841515,-0.2719173541840447,-0.6026066126844256,-0.3301519710411789,-0.006344023570147133,0.43146840602597153,-0.1903930051125798,0.49968384005813127,...,-0.9793172970573618,-0.3934951815221625,0.061522904482061526,-0.4503332997834732,-0.58860886820517,-0.32108496620676896,0.10460116826581342,-0.9956521265152739,-0.10160048285492998,-1.9058324302119607
8,PE,28.47910288725836,-30.29523925978197,0.2742773261982051,0.35198976450909136,28.80586861799048,0.605541561868752,29.119618328275237,1.1608134722209489,28.779272171888667,...,26.783949968445732,28.952094385815528,28.421427923322856,-28.75004278636596,28.78922708751448,-0.8122854253885701,-0.04341117579067182,-0.5853322460832162,-31.081515868896577,-1.3184971894870892
9,SM,1.0335694518443708,-0.42137938586927415,-0.007939915093905727,0.2175818791013914,0.7151266896289435,0.38670111366095916,1.2958916435767787,0.411911450907579,0.39817899578620086,...,-0.12005559835470583,-0.11

In [380]:
# Extraire la colonne "Family" avant conversion
family_column = sum_species.iloc[:, 0]

# Convertir les autres colonnes en float
sum_species_numeric = sum_species.iloc[:, 1:].astype(float)

# Réinsérer la colonne "Family"
sum_species_numeric.insert(0, "Family", family_column)

# Mettre à jour sum_species avec la version corrigée
sum_species = sum_species_numeric
sum_species

,Family,log2FC_D00-BS-064,log2FC_D00-BS-082,log2FC_D00-BS-336,log2FC_D00-BS-346,log2FC_D00-BS-351,log2FC_D00-BS-364,log2FC_D00-BS-377,log2FC_D00-BS-671,log2FC_D00-JV-035,...,log2FC_D00-KT-695,log2FC_D00-KT-705,log2FC_D00-KT-716,log2FC_D00-KT-718,log2FC_D00-KT-723,log2FC_D00-KT-771,log2FC_D00-KT-805,log2FC_D00-KT-880,log2FC_D00-KT-926,log2FC_D00-KT-974
0,DG,-0.422119,-0.070952,-1.531951,-3.493468,0.084956,-1.019564,-0.970436,-1.304212,2.542699,...,-0.307095,-0.142698,0.224062,0.692400,-1.862953,-1.256791,0.636412,-0.821127,2.045362,-1.410360
1,CAR,-0.737824,-0.556003,0.457988,0.367257,-0.239870,0.140086,30.713194,1.189355,29.206879,...,30.009037,-0.558047,1.207687,-1.612445,0.414551,-0.748517,2.108703,-27.305830,1.736387,30.863099
2,CE,2.203710,-0.883650,-0.091410,0.825670,2.371211,1.045849,3.104789,0.138934,-3.025191,...,-0.920261,-0.153927,-2.285419,0.239341,-1.953088,-0.533911,2.293213,-0.310879,-32.181676,1.844886
3,Cer,1.238605,-0.206345,0.658680,1.611261,31.232860,1.440134,2.261833,1.084558,1.565923,...,0.038399,-0.698716,0.610333,0.093508,-0.210835,1.061575,0.051619,1.179902,-0.346314,0.395613
4,Chol,-0.221683,-0.211786,-1.135507,-0.593532,0.526709,0.493887,0.697844,-0.584950,-0.003764,...,-0.630418,0.537567,1.047601,0.219277,-0.503796,0.941027,0.004759,-0.183247,-0.575181,33.180511
5,HexCer,31.141988,-30.795830,0.259910,-28.552595,30.691044,30.278123,29.071785,-29.213511,0.469972,...,-1.131901,-30.877396,-29.116429,-0.236142,-0.248435,30.156599,-29.209688,0.197251,-29.936244,-0.015825
6,LPC,-1.381467,-1.973912,-0.904368,-1.160899,-1.663521,-0.259001,-0.024718,-0.600387,-0.609047,...,-1.463565,-1.440108,-0.345597,-0.819514,-1.055595,-1.016050,0.390271,-0.932435,-0.189193,-2.165814
7,PC,0.296005,-1.296151,-0.271917,-0.602607,-0.330152,-0.006344,0.431468,-0.190393,0.499684,...,-0.979317,-0.393495,0.061523,-0.450333,-0.588609,-0.321085,0.104601,-0.995652,-0.101600,-1.905832
8,PE,28.479103,-30.295239,0.274277,0.351990,28.805869,0.605542,29.119618,1.160813,28.779272,...,26.783950,28.952094,28.421428,-28.750043,28.789227,-0.812285,-0.043411,-0.585332,-31.081516,-1.318497
9,SM,1.033569,-0.421379,-0.007940,0.217582,0.715127,0.386701,1.295892,0.411911,0.398179,...,-0.120056,-0.115348,0.244015,-0.375643,0.210740,0.425726,0.876949,-0.347450,-0.736872,-0.186924


In [381]:
# Calcul du ratio PC/PE
pc_values = sum_species[sum_species["Family"]
                        == "PC"].iloc[:, 1:].values.flatten()
pe_values = sum_species[sum_species["Family"]
                        == "PE"].iloc[:, 1:].values.flatten()

ratios_pc_pe = pc_values / pe_values  # Calcul du ratio PC/PE

In [382]:
ratios_pc_pe

array([ 1.03937633e-02,  4.27839929e-02, -9.91395672e-01, -1.71200039e+00,
       -1.14612746e-02, -1.04766113e-02,  1.48171038e-02, -1.64016881e-01,
        1.73626295e-02, -3.05459697e-02, -8.87902587e-03,  1.20511576e+00,
       -2.98834123e-01, -5.16067297e-03,  2.45050934e-03,  7.51743778e-02,
        1.07904372e+00,  2.69198960e-02,  7.44166864e-02,  8.48321344e-03,
        1.34091755e+00, -2.74620217e+00,  4.02545065e-02,  7.30074292e-01,
       -7.46134833e-01, -1.56986132e+01, -2.71837687e-02,  1.61820520e+00,
       -8.37536728e-01,  2.24517778e+01, -3.65635875e-02, -1.35912510e-02,
        2.16466620e-03,  1.56637436e-02, -2.04454557e-02,  3.95285889e-01,
       -2.40954469e+00,  1.70100338e+00,  3.26883937e-03,  1.44545809e+00])

In [383]:
# Vérifier la longueur des arrays
print(len(patient_info.index), len(ratios_pc_pe))

# Associer les ratios aux échantillons
ratios_df = pd.DataFrame({
    "Sample": patient_info.index[:len(ratios_pc_pe)],
    "Ratio_PC_PE": ratios_pc_pe,
    "Severity": patient_info["Severity"].values[:len(ratios_pc_pe)]
})
ratios_df

40 40


,Sample,Ratio_PC_PE,Severity
0,log2FC_D00-BS-064,0.010394,severe
1,log2FC_D00-BS-082,0.042784,mild
2,log2FC_D00-BS-336,-0.991396,mild
3,log2FC_D00-BS-346,-1.712000,mild
4,log2FC_D00-BS-351,-0.011461,mild
5,log2FC_D00-BS-364,-0.010477,mild
6,log2FC_D00-BS-377,0.014817,mild
7,log2FC_D00-BS-671,-0.164017,mild
8,log2FC_D00-JV-035,0.017363,severe
9,log2FC_D00-JV-138,-0.030546,mild


In [384]:
# Séparer les groupes selon la sévérité
ratios_severe = ratios_df[ratios_df["Severity"] == "severe"]["Ratio_PC_PE"]
ratios_mild = ratios_df[ratios_df["Severity"] == "mild"]["Ratio_PC_PE"]

In [385]:
# Test de normalité (Shapiro-Wilk)
_, p_severe = stats.shapiro(ratios_severe)
_, p_mild = stats.shapiro(ratios_mild)

print(f"P-value normalité Severe: {p_severe:.5f}")
print(f"P-value normalité Mild: {p_mild:.5f}")

P-value normalité Severe: 0.00375
P-value normalité Mild: 0.00000


In [386]:
# Choix du test statistique
if p_severe > 0.05 and p_mild > 0.05:
    # Si les deux groupes sont normalement distribués → test t de Student
    stat, p_value = stats.ttest_ind(ratios_severe, ratios_mild)
    print(f"Test t de Student: stat={stat:.2f}, p={p_value:.5f}")
else:
    # Sinon → test de Mann-Whitney U
    stat, p_value = stats.mannwhitneyu(
        ratios_severe, ratios_mild, alternative="two-sided")
    print(f"Test de Mann-Whitney: stat={stat:.2f}, p={p_value:.5f}")

Test de Mann-Whitney: stat=229.00, p=0.40700


In [387]:
# Test du Chi² (discrétisation en médiane)
median_ratio = ratios_df["Ratio_PC_PE"].median()
ratios_df["High_Low"] = np.where(
    ratios_df["Ratio_PC_PE"] >= median_ratio, "High", "Low")

# Création de la table de contingence et test du Chi²
contingency_table = pd.crosstab(ratios_df["High_Low"], ratios_df["Severity"])
chi2_stat, chi2_p, _, _ = stats.chi2_contingency(contingency_table)

print(f"Test du Chi²: Chi2={chi2_stat:.2f}, p={chi2_p:.5f}")

Test du Chi²: Chi2=0.91, p=0.34036


# D3 - données normalisées (log2FC) avec D60

In [164]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D03.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [165]:
sum_species

,Family,log2FC_D03-BS-064,log2FC_D03-BS-082,log2FC_D03-BS-336,log2FC_D03-BS-346,log2FC_D03-BS-351,log2FC_D03-BS-364,log2FC_D03-BS-377,log2FC_D03-BS-671,log2FC_D03-JV-035,...,log2FC_D03-KT-695,log2FC_D03-KT-705,log2FC_D03-KT-716,log2FC_D03-KT-718,log2FC_D03-KT-723,log2FC_D03-KT-771,log2FC_D03-KT-805,log2FC_D03-KT-880,log2FC_D03-KT-926,log2FC_D03-KT-974
0,DG,0.6443608978399848,-1.3613311661818441,0.24779529708996123,0.6321456246817568,0.9695186812626725,1.1725088929553225,-0.23667823135929844,1.388424413720012,1.037753431218249,...,1.1509550010202863,-1.2229746435253934,0.4580234135524412,0.7946855932456798,1.3374909991208659,-0.7072911267629076,0.32683590538884505,-1.1704143310399155,0.796237147687693,0.5608561343216854
1,CAR,-29.007818160079594,-29.336889823861043,0.07491374337850698,0.1258130142630109,-2.53366635715763,-0.872068594883735,28.671869615969946,-28.263337825440452,31.69393973443677,...,26.965880742183295,-0.2272518913398641,-0.6186954425376907,0.5239473287681591,-0.13587564721510015,-0.5879963322238158,-1.0255756090970471,-27.493872586437494,-27.61036968645289,30.53351056755991
2,CE,0.7577735423199227,-2.5567274501776924,1.2096983990642762,2.3852299104754846,2.795553332233411,-30.300230189092503,-29.355793273377973,1.2666153231476436,-1.131487849369124,...,-0.5688130797705363,1.0556752000623522,-32.47638710993967,1.021071514333942,-33.41140633093559,28.42024928569957,-29.528362228247428,27.739847410020456,-0.5135235632834637,2.6215029273124713
3,Cer,0.8939555733950576,-0.8366753047993482,1.0572768963388703,2.714488379745545,30.703711518701045,1.8156528472670133,0.42558032821242886,1.5233181546177854,0.21181612274190573,...,1.0132854570276806,-1.3806838848655503,0.7971170347471236,0.2346486887603302,-0.8629178382514209,0.6515833748135083,0.6271161490345087,0.6131991747126957,1.5604274029284733,0.990606569598909
4,Chol,-0.6101041631979968,-1.2060273091967284,-0.8990315281676553,-0.8933522773196937,0.6874269267948201,-0.530548509563232,-34.000140607846674,0.62911806501743,-34.599337907389874,...,0.5062829060068342,0.7711116111962655,1.3227485545603979,0.3636894393628511,-0.6655763132935317,0.6689155952937548,-0.3339366723498203,-1.4361459432724697,1.0264932306331056,34.257309241644705
5,HexCer,29.175891141426252,-29.623705087086968,28.98791541444927,-28.540468304182852,30.02166558025188,0.9575416048563941,0.5270792963792527,1.088971971237737,0.8328900141647416,...,-1.4473150959457812,-30.579152256588205,0.25342021303191103,-0.5715235400649706,0.07968240010098673,28.967404011807517,-28.642023316680984,-0.3753567125037256,0.8739919721235587,30.444763555334582
6,LPC,-0.44451988274951204,-1.7430736659666324,-0.6732512873576716,-1.592963635450074,-0.7289746633231041,-0.7428881500030146,0.3986195076531283,-0.00323135889472139,-0.2278121062714535,...,-1.3529741117121243,-0.9876898573217472,-0.5891356757281638,-0.3516994353483104,-0.8368209242094599,0.10866076221996865,-0.004355761626261655,-2.910540117631586,-0.5805746964206281,-0.6214252052861393
7,PC,0.36617919346657307,-1.5199916793235635,-0.31819902540346706,0.545831532881525,-0.022351837618513823,0.9347578048222702,0.3015056080141055,1.0683719556848605,0.23206657588002025,...,-0.05949939326193506,-0.554722745585153,0.1415763427138135,-0.5167841007728728,0.13704240298780712,-0.3755239912649074,-0.2623736162221152,-1.282089732878875,-0.17919416562029225,-1.1063738587724523
8,PE,28.275749473119657,-28.31154789738193,0.3280541976842041,29.57546449573428,0.2532659251585383,-0.318790534036344,0.6600303187875997,29.936624346636002,0.8026657092095529,...,27.25483507066431,0.40030207096800197,-0.304184011990891,-29.707156973512785,28.77344774333207,-0.8626495320596724,-28.794101539066823,-29.548980842835185,-30.19802849573054,27.38810220333949
9,SM,0.10211901329345316,-0.9101186283120054,-0.18197400171196765,-0.27974971648976704,0.47664911744218025,0.2634267507780717,0.4685254898859929,0.4771115246917334,0.11595940169934553,...,0.0029810249187216613,-0.2149562986797385,0.290002

In [166]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [167]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[11:] for col in sample_cols],
    "Day": [col[7:-7] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [168]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D03,log2FC_D03-BS-064,severe
1,BS-082,D03,log2FC_D03-BS-082,mild
2,BS-336,D03,log2FC_D03-BS-336,mild
3,BS-346,D03,log2FC_D03-BS-346,mild
4,BS-351,D03,log2FC_D03-BS-351,mild
5,BS-364,D03,log2FC_D03-BS-364,mild
6,BS-377,D03,log2FC_D03-BS-377,mild
7,BS-671,D03,log2FC_D03-BS-671,mild
8,JV-035,D03,log2FC_D03-JV-035,severe
9,JV-138,D03,log2FC_D03-JV-138,mild


In [169]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [170]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
log2FC_D03-BS-064,BS-064,D03,severe,14,M,severe
log2FC_D03-BS-082,BS-082,D03,mild,18,M,mild
log2FC_D03-BS-336,BS-336,D03,mild,11,F,mild
log2FC_D03-BS-346,BS-346,D03,mild,8,F,mild
log2FC_D03-BS-351,BS-351,D03,mild,7,M,mild
log2FC_D03-BS-364,BS-364,D03,mild,6,M,mild
log2FC_D03-BS-377,BS-377,D03,mild,13,F,mild
log2FC_D03-BS-671,BS-671,D03,mild,8,F,mild
log2FC_D03-JV-035,JV-035,D03,severe,13,M,severe


In [171]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [172]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0     0.794686
1    -0.102722
2    -1.160859
3     0.797903
4     0.144691
5     0.019240
6    -0.589136
7    -0.059499
8     0.341309
9     0.184941
10    1.374195
11    1.201505
12   -0.427091
dtype: float64

In [173]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,log2FC_D03-BS-064,log2FC_D03-BS-082,log2FC_D03-BS-336,log2FC_D03-BS-346,log2FC_D03-BS-351,log2FC_D03-BS-364,log2FC_D03-BS-377,log2FC_D03-BS-671,log2FC_D03-JV-035,log2FC_D03-JV-138,...,log2FC_D03-KT-663,log2FC_D03-KT-695,log2FC_D03-KT-705,log2FC_D03-KT-716,log2FC_D03-KT-718,log2FC_D03-KT-723,log2FC_D03-KT-771,log2FC_D03-KT-805,log2FC_D03-KT-880,log2FC_D03-KT-926
0,0,0,0,0,1,1,0,1,1,0,...,0,1,0,0,0,1,0,0,0,1
1,0,0,1,1,0,0,1,0,1,0,...,0,1,0,0,1,0,0,0,0,0
2,1,0,1,1,1,0,0,1,1,0,...,0,1,1,0,1,0,1,0,1,1
3,1,0,1,1,1,1,0,1,0,1,...,1,1,0,0,0,0,0,0,0,1
4,0,0,0,0,1,0,0,1,0,0,...,1,1,1,1,1,0,1,0,0,1
5,1,0,1,0,1,1,1,1,1,1,...,0,0,0,1,0,1,1,0,0,1
6,1,0,0,0,0,0,1,1,1,0,...,0,0,0,0,1,0,1,1,0,1
7,1,0,0,1,1,1,1,1,1,0,...,1,0,0,1,0,1,0,0,0,0
8,1,0,0,1,0,0,1,1,1,1,...,1,1,1,0,0,1,0,0,0,0
9,0,0,0,0,1,1,1,1,0,0,...,1,0,0,1,0,1,1,1,0,1


## Test du Chi² sur les lipides en fonction de la sévérité

In [174]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [175]:
chi2_results

{'DG': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'CAR': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'CE': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'Cer': {'Chi2': 0.0, 'p-value': 1.0},
 'Chol': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'HexCer': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'LPC': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046},
 'PC': {'Chi2': 0.0, 'p-value': 1.0},
 'PE': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'SM': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'TG': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'FA': {'Chi2': 0.0, 'p-value': 1.0},
 'PI': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303}}

LPC montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides seraient liés à la gravité de la maladie. Reste à voir si cette valeur passe la correction de Benjamini-Hochberg.

In [176]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [177]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_log2FC_D03.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [178]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

            Chi2   p-value  p-value (BH corrected)
DG      1.237077  0.266035                0.576409
CAR     0.220536  0.638632                0.830222
CE      1.237077  0.266035                0.576409
Cer     0.000000  1.000000                1.000000
Chol    1.237077  0.266035                0.576409
HexCer  0.665273  0.414705                0.673895
LPC     5.747979  0.016508                0.214599
PC      0.000000  1.000000                1.000000
PE      0.220536  0.638632                0.830222
SM      0.665273  0.414705                0.673895
TG      3.079558  0.079282                0.343556
FA      0.000000  1.000000                1.000000
PI      3.079558  0.079282                0.343556


In [179]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_log2FC_D03.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv


# D10 - données normalisées (log2FC) avec D60

In [185]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D10.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [186]:
sum_species

,Family,log2FC_D10-BS-064,log2FC_D10-BS-082,log2FC_D10-BS-336,log2FC_D10-BS-346,log2FC_D10-BS-351,log2FC_D10-BS-364,log2FC_D10-BS-377,log2FC_D10-BS-671,log2FC_D10-JV-035,...,log2FC_D10-KT-695,log2FC_D10-KT-705,log2FC_D10-KT-716,log2FC_D10-KT-718,log2FC_D10-KT-723,log2FC_D10-KT-771,log2FC_D10-KT-805,log2FC_D10-KT-880,log2FC_D10-KT-926,log2FC_D10-KT-974
0,DG,0.42435250260464386,0.15739391544177705,0.30051397331989665,1.2947517463779137,1.3731267334323396,1.1254796129664864,-0.3847683174327007,-0.1207729733368489,1.6130184522541817,...,1.546650387719339,-0.6365566955925968,-1.6653432060664997,1.5631318863545842,1.1085644540355002,-0.3446726188452463,1.405524267559222,0.5481054324022636,0.8184361907703342,0.0034342893141425554
1,CAR,-28.53349257725569,-2.524822547756122,-2.2806520390940523,-0.14548873301595508,-30.3721309843492,-0.24890989597198357,27.886412535661723,-1.440226460643737,0.47257928352063866,...,29.022450261704957,-1.06874943999376,-2.8353384171434604,-2.27465432615388,-29.999185103074378,-1.5683352177871863,-1.8494955994859228,1.2545459877556402,-27.465305447743564,0.45909442217885216
2,CE,-29.97907327205604,-0.6035267473313587,1.3916253179070914,-30.02488435183628,-0.13605940409932618,0.2734641264428535,-0.874963595565858,0.10034826995410448,-31.675841672652314,...,-3.5256014838756524,-1.45186209781572,-0.8509522729070254,0.47207955058278694,-1.5936554035559085,31.327238571839484,0.45444208620269794,0.05215387914552844,-31.954395832346503,-29.58043286315923
3,Cer,0.3162153783852671,-0.6956268538328882,1.6150196957488754,0.8728253096658338,30.532203732032624,0.20858041091410995,1.8601356234788735,-0.020256106488753985,0.5603967103667798,...,0.039257663909539116,-1.0307022394822367,0.41172627287431335,0.43634544880092685,0.6921273968941012,-0.14721488804533872,0.07142412956032762,1.2327001271402565,0.5791555755495776,0.9641768156948388
4,Chol,-0.8623135274353392,-1.2023316908718558,-1.106104725040929,-1.4541369732647915,0.5651493271280745,-0.905413872541553,0.3164336958620363,0.2670361784212605,-34.70907549502498,...,-0.014619162551375506,0.34678550801228464,1.3074406510273293,0.8190497039988913,0.13129962370666168,1.3447736956309642,-0.36924079802767773,0.13923474383077566,-0.24568798750904353,34.34231458615274
5,HexCer,0.4144667000465158,-30.43840198448574,29.235671496713586,-28.860934290351185,-0.1786784869826052,29.863965760079466,0.2914992902201895,-29.312473156587092,1.1057258872942979,...,-1.2006483014996092,-30.66454912656381,-29.371224138537126,28.287657538353045,29.201038552835552,28.52918518756093,-29.196029004851695,-0.046078009697783855,-29.045061232490255,0.5107208473341514
6,LPC,0.3362873417807732,-0.9581596449546744,-0.050098160731386746,-0.2675104206377734,-0.3956469799002727,0.668581313560072,0.6213214438454228,0.6110824261959892,-0.4281602328377818,...,0.6228145034103818,-0.18247481805853066,-0.5176156716753623,0.26316376484706244,0.05970996413834274,-0.10005843007192737,0.42978888599904297,0.04848657135534054,0.383816146048294,-0.24033823596223558
7,PC,0.8878584347931494,-1.004788909673796,0.3176480143009372,0.954784330886762,0.2954785006673525,0.7220095528113513,0.7063003105672777,0.18072515229538358,0.4782089875606258,...,0.390998958010769,0.2822846317980038,0.33494472214080717,0.7401644934390005,0.21110055600892239,0.2860516828565608,0.034721501311051804,0.15951414698351052,0.026463186323781824,-0.3669148963065245
8,PE,28.71597104318901,-29.63931023770667,0.3112696133886405,29.407534893865087,-0.19119440407367064,-28.522433036674393,29.86933083574041,1.1339505720655898,0.6385954993827029,...,27.134668673719894,28.3844897566995,28.98454085017421,0.4095236102115718,-0.35097268345836147,0.27171734953363813,-29.40769242408211,0.21743132667768714,-29.786191133503845,-0.47567292388724675
9,SM,0.07799497819235884,-1.296984409196567,-0.21006251951780627,-0.4314545363279268,-0.09201453291841899,0.34127349969648535,0.4936804134528652,0.2541812835657561,-0.0603188202670243,...,-0.4850780509949868,-0.26853

In [187]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [188]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[11:] for col in sample_cols],
    "Day": [col[7:-7] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [189]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D10,log2FC_D10-BS-064,severe
1,BS-082,D10,log2FC_D10-BS-082,mild
2,BS-336,D10,log2FC_D10-BS-336,mild
3,BS-346,D10,log2FC_D10-BS-346,mild
4,BS-351,D10,log2FC_D10-BS-351,mild
5,BS-364,D10,log2FC_D10-BS-364,mild
6,BS-377,D10,log2FC_D10-BS-377,mild
7,BS-671,D10,log2FC_D10-BS-671,mild
8,JV-035,D10,log2FC_D10-JV-035,severe
9,JV-138,D10,log2FC_D10-JV-138,mild


In [190]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [191]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
log2FC_D10-BS-064,BS-064,D10,severe,14,M,severe
log2FC_D10-BS-082,BS-082,D10,mild,18,M,mild
log2FC_D10-BS-336,BS-336,D10,mild,11,F,mild
log2FC_D10-BS-346,BS-346,D10,mild,8,F,mild
log2FC_D10-BS-351,BS-351,D10,mild,7,M,mild
log2FC_D10-BS-364,BS-364,D10,mild,6,M,mild
log2FC_D10-BS-377,BS-377,D10,mild,13,F,mild
log2FC_D10-BS-671,BS-671,D10,mild,8,F,mild
log2FC_D10-JV-035,JV-035,D10,severe,13,M,severe


In [192]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [193]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0     1.095902
1    -0.567530
2    -0.603527
3     0.560397
4     0.247852
5    -0.046078
6     0.287138
7     0.390999
8     0.404924
9     0.025576
10    1.189445
11    0.631031
12    0.437379
dtype: float64

In [194]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,log2FC_D10-BS-064,log2FC_D10-BS-082,log2FC_D10-BS-336,log2FC_D10-BS-346,log2FC_D10-BS-351,log2FC_D10-BS-364,log2FC_D10-BS-377,log2FC_D10-BS-671,log2FC_D10-JV-035,log2FC_D10-JV-138,...,log2FC_D10-KT-663,log2FC_D10-KT-695,log2FC_D10-KT-705,log2FC_D10-KT-716,log2FC_D10-KT-718,log2FC_D10-KT-723,log2FC_D10-KT-771,log2FC_D10-KT-805,log2FC_D10-KT-880,log2FC_D10-KT-926
0,0,0,0,1,1,1,0,0,1,1,...,1,1,0,0,1,1,0,1,0,0
1,0,0,0,1,0,1,1,0,1,0,...,1,1,0,0,0,0,0,0,1,0
2,0,0,1,0,1,1,0,1,0,0,...,0,0,0,0,1,0,1,1,1,0
3,0,0,1,1,1,0,1,0,0,1,...,1,0,0,0,0,1,0,0,1,1
4,0,0,0,0,1,0,1,1,0,0,...,1,0,1,1,1,0,1,0,0,0
5,1,0,1,0,0,1,1,0,1,1,...,1,0,0,0,1,1,1,0,0,0
6,1,0,0,0,0,1,1,1,0,1,...,1,1,0,0,0,0,0,1,0,1
7,1,0,0,1,0,1,1,0,1,1,...,1,0,0,0,1,0,0,0,0,0
8,1,0,0,1,0,0,1,1,1,1,...,0,1,1,1,1,0,0,0,0,0
9,1,0,0,0,0,1,1,1,0,0,...,1,0,0,1,0,1,0,0,1,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [195]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [196]:
chi2_results

{'DG': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'CE': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'Cer': {'Chi2': 0.0, 'p-value': 1.0},
 'Chol': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'HexCer': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'LPC': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PC': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'PE': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'SM': {'Chi2': 0.0, 'p-value': 1.0},
 'TG': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'FA': {'Chi2': 0.0, 'p-value': 1.0},
 'PI': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848}}

Aucun lipide ne montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides ne seraient pas liés à la gravité de la maladie.

In [ ]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [ ]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_log2FC_D03.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [ ]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

            Chi2   p-value  p-value (BH corrected)
DG      1.237077  0.266035                0.576409
CAR     0.220536  0.638632                0.830222
CE      1.237077  0.266035                0.576409
Cer     0.000000  1.000000                1.000000
Chol    1.237077  0.266035                0.576409
HexCer  0.665273  0.414705                0.673895
LPC     5.747979  0.016508                0.214599
PC      0.000000  1.000000                1.000000
PE      0.220536  0.638632                0.830222
SM      0.665273  0.414705                0.673895
TG      3.079558  0.079282                0.343556
FA      0.000000  1.000000                1.000000
PI      3.079558  0.079282                0.343556


In [ ]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_log2FC_D03.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv
